# EDA — Índice Canasta Atlas

Análisis exploratorio de la serie de precios de la Canasta Atlas (26 productos, Coto Digital).

**Regla de oro del índice:** se construye con `precio_lista`. El campo `precio_promo` se dejó fuera del índice — históricamente la API de Coto devolvía valores absurdos (ver sección final) y, aun corregido, un promo es un precio puntual/condicional que no refleja el costo estructural de la canasta.

Este notebook se regenera solo a medida que entra data diaria: no hay valores hardcodeados.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)

# Ruta a la DB relativa a la raíz del repo (funciona corriendo desde notebooks/ o raíz)
DB = Path("data/atlas.db")
if not DB.exists():
    DB = Path("../data/atlas.db")

con = sqlite3.connect(DB)
df = pd.read_sql(
    """
    SELECT pr.fecha, p.nombre_normalizado AS nombre, p.categoria,
           pr.precio_lista, pr.precio_promo
    FROM precios pr
    JOIN productos p ON p.id = pr.producto_id
    WHERE p.en_canasta = 1
    ORDER BY pr.fecha
    """,
    con,
)
con.close()

fechas = sorted(df["fecha"].unique())
print(f"{len(fechas)} días: {fechas[0]} → {fechas[-1]}")
print(f"{df['nombre'].nunique()} productos de canasta")
df.head()

## 1. Índice Canasta Atlas (base 100)

Costo de comprar la misma canasta fija cada día, expresado en base 100 al primer día. Para comparar peras con peras se usan solo los productos con serie completa en todo el período.

In [ ]:
piv = df.pivot_table(index="fecha", columns="nombre", values="precio_lista").sort_index()
completos = piv.dropna(axis=1)  # productos presentes todos los días

canasta = completos.sum(axis=1)
indice = canasta / canasta.iloc[0] * 100

base, ult = piv.index[0], piv.index[-1]
var = indice.iloc[-1] - 100
dias = (pd.to_datetime(ult) - pd.to_datetime(base)).days

print(f"Serie completa: {completos.shape[1]}/{piv.shape[1]} productos")
print(indice.round(3).to_string())
print(f"\nVariación {base} → {ult}: {var:+.2f}%")
print(f"Costo canasta: ${canasta.iloc[0]:,.0f} → ${canasta.iloc[-1]:,.0f}")
if dias:
    print(f"Proyección mensual (30d): {var / dias * 30:+.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(indice.index, indice.values, marker="o")
ax.axhline(100, color="grey", ls="--", lw=0.8)
ax.set_title("Índice Canasta Atlas — base 100")
ax.set_ylabel("Índice")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

## 2. Variación por categoría

In [ ]:
filas = []
for cat, g in df.groupby("categoria"):
    p = g.pivot_table(index="fecha", columns="nombre", values="precio_lista").dropna(axis=1)
    if p.shape[1] == 0:
        continue
    s = p.sum(axis=1)
    filas.append({
        "categoria": cat,
        "n_prod": p.shape[1],
        "costo_base": s.iloc[0],
        "costo_ult": s.iloc[-1],
        "var_%": (s.iloc[-1] / s.iloc[0] - 1) * 100,
    })

cat_df = pd.DataFrame(filas).sort_values("var_%", ascending=False)
cat_df.round(2)

## 3. Mayores subas y bajas por producto

In [ ]:
mov = ((piv.loc[ult] - piv.loc[base]) / piv.loc[base] * 100).dropna().sort_values(ascending=False)
print("TOP SUBAS (%)")
print(mov.head(5).round(1).to_string())
print("\nTOP BAJAS (%)")
print(mov.tail(5).round(1).to_string())

## 4. Control de calidad: el bug histórico de `precio_promo`

Antes del fix del scraper (`scraper/coto.py` → `_extraer_precio_promo`), el parser inflaba el `discountPrice` de Coto y guardaba promos absurdos (>10x el precio de lista). Un promo válido **siempre** es menor al precio de lista, así que ese es el guard de sanidad que se aplica en el scraper y en `pipeline/normalize.py`.

Esta celda confirma que la DB actual está limpia (debe dar 0).

In [ ]:
absurdos = df[df["precio_promo"].notna() & (df["precio_promo"] >= df["precio_lista"])]
print(f"Registros con promo >= precio_lista: {len(absurdos)}  (esperado: 0)")
con_promo = df["precio_promo"].notna().sum()
print(f"Registros con promo válido: {con_promo}")